# Merchant-Level Feature Pipeline (Crash-Safe, Data_1 Only)

This notebook is a safe orchestrator for the merchant pipeline.

Why this notebook is different:
- heavy Polars work does not run inside the Jupyter kernel
- every heavy stage runs in a separate subprocess
- the notebook kernel stays light and stable
- all intermediate layers are materialized to parquet and reused
- the source is only `data/Data_1`

Crash-safe design:
- Stage 1 writes prepared transaction parquet shards
- pair-heavy stages are split into shard partial counts and merge stages
- category, MCC, daily, and merchant-user features are built from aggregated parquet layers
- thread count is capped from the notebook to reduce memory spikes

Important note:
- `avg_unique_users_per_active_day` is intentionally left null in this crash-safe version
- exact computation of that feature requires a very large `(merchant, day, user)` layer and is intentionally skipped to avoid kernel/process crashes on the current data volume


## 1. Imports and Runtime Config

This cell:
- defines the runtime script path
- fixes the source to `Data_1`
- sets conservative thread limits for Polars subprocesses
- configures rebuild flags


In [1]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

import polars as pl

ROOT = Path('.')
RUNTIME_SCRIPT_PATH = ROOT / 'merchant_pipeline_runtime.py'
DATA_DIR = ROOT / 'data' / 'Data_1'
ARTIFACTS_DIR = ROOT / 'artifacts'
INTERMEDIATE_DIR = ARTIFACTS_DIR / 'intermediate'
FEATURES_DIR = INTERMEDIATE_DIR / 'merchant_feature_stages'
PIPELINE_VERSION = 'merchant_v6_crash_safe'

FINAL_PARQUET_PATH = ARTIFACTS_DIR / f'merchant_features_data1_{PIPELINE_VERSION}.parquet'
FINAL_CSV_PATH = ARTIFACTS_DIR / f'merchant_features_data1_{PIPELINE_VERSION}.csv'
BUCKET_CONFIG_PATH = INTERMEDIATE_DIR / f'merchant_bucket_config_{PIPELINE_VERSION}.json'
PREPARED_SHARDS_DIR = INTERMEDIATE_DIR / f'merchant_prepared_transactions_{PIPELINE_VERSION}_shards'
DAILY_COUNT_SHARDS_DIR = INTERMEDIATE_DIR / f'merchant_daily_counts_{PIPELINE_VERSION}_shards'
CATEGORY_COUNT_SHARDS_DIR = INTERMEDIATE_DIR / f'merchant_category_counts_{PIPELINE_VERSION}_shards'
MCC_COUNT_SHARDS_DIR = INTERMEDIATE_DIR / f'merchant_mcc_counts_{PIPELINE_VERSION}_shards'
MERCHANT_USER_SHARDS_DIR = INTERMEDIATE_DIR / f'merchant_user_counts_{PIPELINE_VERSION}_shards'

REBUILD_PREPARED = False
REBUILD_BUCKETS = False
REBUILD_FEATURE_STAGES = False

DEFAULT_MAX_THREADS = max(1, min(4, os.cpu_count() or 4))
POLARS_MAX_THREADS = DEFAULT_MAX_THREADS

print({
    'runtime_script_exists': RUNTIME_SCRIPT_PATH.exists(),
    'data_dir': str(DATA_DIR.resolve()),
    'pipeline_version': PIPELINE_VERSION,
    'polars_max_threads': POLARS_MAX_THREADS,
    'rebuild_prepared': REBUILD_PREPARED,
    'rebuild_buckets': REBUILD_BUCKETS,
    'rebuild_feature_stages': REBUILD_FEATURE_STAGES,
})


{'runtime_script_exists': True, 'data_dir': 'C:\\PROD2026\\ThirdStageCommand\\data\\Data_1', 'pipeline_version': 'merchant_v6_crash_safe', 'polars_max_threads': 4, 'rebuild_prepared': False, 'rebuild_buckets': False, 'rebuild_feature_stages': False}


## 2. Subprocess Runner Helpers

This cell:
- runs each heavy stage in an external Python process
- caps Polars threads for memory stability
- keeps the notebook kernel almost idle during heavy processing
- provides lightweight artifact preview helpers


In [2]:
def build_stage_env() -> dict[str, str]:
    env = os.environ.copy()
    env['POLARS_MAX_THREADS'] = str(POLARS_MAX_THREADS)
    env.setdefault('PYTHONUTF8', '1')
    return env


def run_stage(stage: str, rebuild: bool = False) -> None:
    cmd = [sys.executable, str(RUNTIME_SCRIPT_PATH), stage]
    if rebuild:
        cmd.append('--rebuild')
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=str(ROOT), env=build_stage_env())


def list_stage_artifacts() -> dict[str, list[str]]:
    return {
        'prepared_shards': sorted(p.name for p in PREPARED_SHARDS_DIR.glob('part_*.parquet'))[:10],
        'daily_count_shards': sorted(p.name for p in DAILY_COUNT_SHARDS_DIR.glob('part_*.parquet'))[:10],
        'category_count_shards': sorted(p.name for p in CATEGORY_COUNT_SHARDS_DIR.glob('part_*.parquet'))[:10],
        'mcc_count_shards': sorted(p.name for p in MCC_COUNT_SHARDS_DIR.glob('part_*.parquet'))[:10],
        'merchant_user_shards': sorted(p.name for p in MERCHANT_USER_SHARDS_DIR.glob('part_*.parquet'))[:10],
        'final_parquet_exists': FINAL_PARQUET_PATH.exists(),
        'final_csv_exists': FINAL_CSV_PATH.exists(),
    }


list_stage_artifacts()


{'prepared_shards': [],
 'daily_count_shards': [],
 'category_count_shards': [],
 'mcc_count_shards': [],
 'merchant_user_shards': [],
 'final_parquet_exists': False,
 'final_csv_exists': False}

## 3. Stage 1 - Prepared Transaction Shards

This stage:
- reads raw CSV only from `Data_1`
- normalizes merchant names
- keeps only needed columns
- writes prepared parquet shards

Why it is safer:
- no giant output dataframe is collected into the notebook kernel
- shard reuse means you almost never need to recompute raw CSV parsing


In [5]:
run_stage('prepare_shards', rebuild=REBUILD_PREPARED)
list_stage_artifacts()


Running: c:\Users\user\AppData\Local\Programs\Python\Python312\python.exe merchant_pipeline_runtime.py prepare_shards


{'prepared_shards': ['part_000.parquet',
  'part_001.parquet',
  'part_002.parquet',
  'part_003.parquet'],
 'daily_count_shards': [],
 'category_count_shards': [],
 'mcc_count_shards': [],
 'merchant_user_shards': [],
 'final_parquet_exists': False,
 'final_csv_exists': False}

## 4. Top-N Bucket Config

This stage:
- computes global top categories and top MCC labels
- saves the bucket config to JSON
- is reused by category and MCC feature stages


In [6]:
run_stage('top_buckets', rebuild=REBUILD_BUCKETS)
json.loads(BUCKET_CONFIG_PATH.read_text(encoding='utf-8'))


Running: c:\Users\user\AppData\Local\Programs\Python\Python312\python.exe merchant_pipeline_runtime.py top_buckets


{'top_category_values': ['Супермаркеты',
  'Фаст Фуд',
  'Транспорт',
  'Разные товары',
  'Топливо',
  'Рестораны',
  'Аптеки',
  'Дом/Ремонт',
  'Развлечения',
  'Сервисные услуги',
  'Автоуслуги',
  'Связь/Телеком',
  'Одежда/Обувь',
  'Красота',
  'Медицинские услуги',
  'Госсборы',
  'Аренда авто',
  'Частные услуги',
  'НКО',
  'Финансовые услуги',
  'Цветы',
  'Книги',
  'Животные',
  'Образование',
  'Кино',
  'Турагентства',
  'Ж/д билеты',
  'Авиабилеты',
  'Спорттовары',
  'unknown_category'],
 'category_share_values': ['Супермаркеты',
  'Фаст Фуд',
  'Транспорт',
  'Разные товары',
  'Топливо',
  'Рестораны',
  'Аптеки',
  'Дом/Ремонт',
  'Развлечения',
  'Сервисные услуги',
  'Автоуслуги',
  'Связь/Телеком',
  'Одежда/Обувь',
  'Красота',
  'Медицинские услуги',
  'Госсборы',
  'Аренда авто',
  'Частные услуги',
  'НКО',
  'Финансовые услуги',
  'Цветы',
  'Книги',
  'Животные',
  'Образование',
  'Кино',
  'Турагентства',
  'Ж/д билеты',
  'Авиабилеты',
  'Спорттовары',
 

## 5. Stage 2 - Merchant Base Features

This stage builds merchant-level base activity and time behavior from prepared parquet shards.

Included here:
- merchant transaction count
- exact unique transaction count by merchant
- first and last transaction time
- weekend and weekday shares
- daypart shares
- dominant hour
- dominant weekday number
- dominant daypart


In [7]:
run_stage('base_features', rebuild=REBUILD_FEATURE_STAGES)


Running: c:\Users\user\AppData\Local\Programs\Python\Python312\python.exe merchant_pipeline_runtime.py base_features


## 6. Stage 3a - Merchant Daily Count Shards

This stage creates shard-level `(merchant, day)` transaction counts.

Why this split matters:
- daily aggregation is one of the most memory-sensitive parts
- partial shard counts reduce peak memory before the merge step


In [8]:
run_stage('daily_count_shards', rebuild=REBUILD_FEATURE_STAGES)
list_stage_artifacts()


Running: c:\Users\user\AppData\Local\Programs\Python\Python312\python.exe merchant_pipeline_runtime.py daily_count_shards


{'prepared_shards': ['part_000.parquet',
  'part_001.parquet',
  'part_002.parquet',
  'part_003.parquet'],
 'daily_count_shards': ['part_000.parquet',
  'part_001.parquet',
  'part_002.parquet',
  'part_003.parquet'],
 'category_count_shards': [],
 'mcc_count_shards': [],
 'merchant_user_shards': [],
 'final_parquet_exists': False,
 'final_csv_exists': False}

## 7. Stage 3b and 3c - Merchant Daily Features

This stage merges daily count shards and produces:
- active days count
- active weeks count
- active months count
- max transactions in one day
- median transactions per active day


In [9]:
run_stage('daily_features', rebuild=REBUILD_FEATURE_STAGES)


Running: c:\Users\user\AppData\Local\Programs\Python\Python312\python.exe merchant_pipeline_runtime.py daily_features


## 8. Stage 4a - Merchant Category Count Shards

This stage creates shard-level `(merchant, category)` counts.

Why this is safer:
- category distribution is computed from aggregated counts, not from the full raw transaction layer every time


In [10]:
run_stage('category_count_shards', rebuild=REBUILD_FEATURE_STAGES)
list_stage_artifacts()


Running: c:\Users\user\AppData\Local\Programs\Python\Python312\python.exe merchant_pipeline_runtime.py category_count_shards


{'prepared_shards': ['part_000.parquet',
  'part_001.parquet',
  'part_002.parquet',
  'part_003.parquet'],
 'daily_count_shards': ['part_000.parquet',
  'part_001.parquet',
  'part_002.parquet',
  'part_003.parquet'],
 'category_count_shards': ['part_000.parquet',
  'part_001.parquet',
  'part_002.parquet',
  'part_003.parquet'],
 'mcc_count_shards': [],
 'merchant_user_shards': [],
 'final_parquet_exists': False,
 'final_csv_exists': False}

## 9. Stage 4b and 4c - Merchant Category Features

This stage merges category counts and produces:
- unique category count
- top category transaction share
- category entropy
- top-N category share features with `other`


In [11]:
run_stage('category_features', rebuild=REBUILD_FEATURE_STAGES)


Running: c:\Users\user\AppData\Local\Programs\Python\Python312\python.exe merchant_pipeline_runtime.py category_features


## 10. Stage 5a - Merchant MCC Count Shards

This stage creates shard-level `(merchant, MCC)` counts.


In [12]:
run_stage('mcc_count_shards', rebuild=REBUILD_FEATURE_STAGES)
list_stage_artifacts()


Running: c:\Users\user\AppData\Local\Programs\Python\Python312\python.exe merchant_pipeline_runtime.py mcc_count_shards


{'prepared_shards': ['part_000.parquet',
  'part_001.parquet',
  'part_002.parquet',
  'part_003.parquet'],
 'daily_count_shards': ['part_000.parquet',
  'part_001.parquet',
  'part_002.parquet',
  'part_003.parquet'],
 'category_count_shards': ['part_000.parquet',
  'part_001.parquet',
  'part_002.parquet',
  'part_003.parquet'],
 'mcc_count_shards': ['part_000.parquet',
  'part_001.parquet',
  'part_002.parquet',
  'part_003.parquet'],
 'merchant_user_shards': [],
 'final_parquet_exists': False,
 'final_csv_exists': False}

## 11. Stage 5b and 5c - Merchant MCC Features

This stage merges MCC counts and produces:
- unique MCC count
- top MCC transaction share
- MCC entropy
- top-N MCC share features with `other`


In [13]:
run_stage('mcc_features', rebuild=REBUILD_FEATURE_STAGES)


Running: c:\Users\user\AppData\Local\Programs\Python\Python312\python.exe merchant_pipeline_runtime.py mcc_features


## 12. Stage 6a - Merchant-User Count Shards

This stage creates shard-level `(merchant, user)` transaction counts.

Why this split matters:
- exact user interaction features are expensive on large data
- shard-level partial counts keep the notebook safe and reduce peak memory in each subprocess


In [14]:
run_stage('merchant_user_count_shards', rebuild=REBUILD_FEATURE_STAGES)
list_stage_artifacts()


Running: c:\Users\user\AppData\Local\Programs\Python\Python312\python.exe merchant_pipeline_runtime.py merchant_user_count_shards


{'prepared_shards': ['part_000.parquet',
  'part_001.parquet',
  'part_002.parquet',
  'part_003.parquet'],
 'daily_count_shards': ['part_000.parquet',
  'part_001.parquet',
  'part_002.parquet',
  'part_003.parquet'],
 'category_count_shards': ['part_000.parquet',
  'part_001.parquet',
  'part_002.parquet',
  'part_003.parquet'],
 'mcc_count_shards': ['part_000.parquet',
  'part_001.parquet',
  'part_002.parquet',
  'part_003.parquet'],
 'merchant_user_shards': ['part_000.parquet',
  'part_001.parquet',
  'part_002.parquet',
  'part_003.parquet'],
 'final_parquet_exists': False,
 'final_csv_exists': False}

## 13. Stage 6b and 6c - Merchant User Interaction Features

This stage merges merchant-user counts and produces:
- exact unique user count
- average, median, and max transactions per user
- top user transaction share
- share of single-visit users
- user concentration ratio


In [15]:
run_stage('user_features', rebuild=REBUILD_FEATURE_STAGES)


Running: c:\Users\user\AppData\Local\Programs\Python\Python312\python.exe merchant_pipeline_runtime.py user_features


## 14. Final Stage - Assemble Merchant Features

This stage joins all merchant-level feature layers by `normalized_merchant_nm`, writes final parquet and CSV, and runs final sanity checks.


In [16]:
run_stage('final', rebuild=REBUILD_FEATURE_STAGES)


Running: c:\Users\user\AppData\Local\Programs\Python\Python312\python.exe merchant_pipeline_runtime.py final


## 15. Final Sanity View

This cell is lightweight and safe to run in the notebook kernel because it only reads the final merchant table.


In [17]:
final_df = pl.read_parquet(FINAL_PARQUET_PATH)

print('shape:', final_df.shape)
print('merchant_count:', final_df.height)
print('first_columns:', final_df.columns[:20])
final_df.head(10)


shape: (1213627, 102)
merchant_count: 1213627
first_columns: ['normalized_merchant_nm', 'merchant_txn_cnt', 'unique_transaction_cnt', 'first_txn_dttm', 'last_txn_dttm', 'activity_span_days', 'share_weekend_txn', 'share_weekday_txn', 'share_night_txn', 'share_morning_txn', 'share_afternoon_txn', 'share_evening_txn', 'dominant_hour', 'dominant_weekday_num', 'dominant_daypart', 'active_days_cnt', 'active_weeks_cnt', 'active_months_cnt', 'max_txn_in_one_day', 'median_txn_per_active_day']


normalized_merchant_nm,merchant_txn_cnt,unique_transaction_cnt,first_txn_dttm,last_txn_dttm,activity_span_days,share_weekend_txn,share_weekday_txn,share_night_txn,share_morning_txn,share_afternoon_txn,share_evening_txn,dominant_hour,dominant_weekday_num,dominant_daypart,active_days_cnt,active_weeks_cnt,active_months_cnt,max_txn_in_one_day,median_txn_per_active_day,unique_categories_cnt,top_category_txn_share,category_entropy,merchant_cat_share_супермаркеты,merchant_cat_share_фаст_фуд,merchant_cat_share_транспорт,merchant_cat_share_разные_товары,merchant_cat_share_топливо,merchant_cat_share_рестораны,merchant_cat_share_аптеки,merchant_cat_share_дом_ремонт,merchant_cat_share_развлечения,merchant_cat_share_сервисные_услуги,merchant_cat_share_автоуслуги,merchant_cat_share_связь_телеком,merchant_cat_share_одежда_обувь,merchant_cat_share_красота,…,merchant_mcc_share_5812,merchant_mcc_share_4111,merchant_mcc_share_5300,merchant_mcc_share_4121,merchant_mcc_share_5200,merchant_mcc_share_7995,merchant_mcc_share_5331,merchant_mcc_share_5993,merchant_mcc_share_7542,merchant_mcc_share_5964,merchant_mcc_share_4900,merchant_mcc_share_5816,merchant_mcc_share_3991,merchant_mcc_share_5815,merchant_mcc_share_5311,merchant_mcc_share_5462,merchant_mcc_share_7512,merchant_mcc_share_4814,merchant_mcc_share_4816,merchant_mcc_share_5399,merchant_mcc_share_5977,merchant_mcc_share_5533,merchant_mcc_share_unknown,merchant_mcc_share_other,unique_users_cnt,median_txn_per_user,max_txn_per_user,top_user_txn_share,share_of_single_visit_users,avg_txn_per_user,user_concentration_ratio,avg_txn_per_active_day,avg_txn_per_active_week,avg_txn_per_active_month,dominant_weekday,avg_unique_users_per_active_day,merchant_popularity_rank_proxy
str,u32,u32,datetime[μs],datetime[μs],i32,f64,f64,f64,f64,f64,f64,i8,i8,str,u32,u32,u32,u32,f64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,f64,u32,f64,f64,f64,f64,f64,f64,f64,str,f64,u32
"""krasnoe&beloe""",1460618,1460618,2025-02-01 00:21:17,2025-02-28 22:59:46,27,0.28997,0.71003,0.003051,0.16801,0.498184,0.330755,18,5,"""afternoon""",28,5,1,63480,50430.5,1,1.0,0.0,6.8464e-7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,306595,2.0,131,0.00009,0.339255,4.763998,0.209908,52164.928571,292123.6,1.460618e6,"""Friday""",null,1
"""yandex*4121*taxi""",962037,962037,2025-02-01 00:00:00,2025-02-28 23:59:54,27,0.290154,0.709846,0.089528,0.273846,0.346751,0.289876,18,6,"""afternoon""",28,5,1,39859,33490.5,2,0.999978,0.000256,0.0,0.0,0.000001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.000001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,129232,4.0,597,0.000621,0.214668,7.444263,0.134332,34358.464286,192407.4,962037.0,"""Saturday""",null,2
"""yandexgo""",750598,750598,2025-02-01 00:00:01,2025-02-28 23:59:57,27,0.2941,0.7059,0.08899,0.265488,0.344661,0.30086,18,6,"""afternoon""",28,5,1,31258,26051.0,1,1.0,0.0,0.0,0.0,0.000001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.000001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,104839,4.0,300,0.0004,0.217123,7.15953,0.139674,26807.071429,150119.6,750598.0,"""Saturday""",null,3
"""mos.transport""",596827,596827,2025-02-01 00:00:00,2025-02-28 23:59:16,27,0.223643,0.776357,0.01535,0.358248,0.332009,0.294394,11,5,"""morning""",28,5,1,25204,22275.5,1,1.0,0.0,0.0,0.0,0.000002,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.000002,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,88543,3.0,159,0.000266,0.209186,6.740533,0.148356,21315.25,119365.4,596827.0,"""Friday""",null,4
"""wildberries""",544638,544638,2025-02-01 00:00:00,2025-02-28 23:59:54,27,0.30178,0.69822,0.035177,0.21172,0.452688,0.300414,18,7,"""afternoon""",28,5,1,26587,24159.0,5,

## 16. Optional Single-Command Full Run

Use this only when you want the whole pipeline in one subprocess.

It is still safer than a pure notebook run because the heavy work stays outside the kernel.


In [18]:
# run_stage('all', rebuild=False)
